In [ ]:
%matplotlib inline

# 99.3 Reactions

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Object creation, stoichiometric lists, rate parameters and simulation calls.

**Introduces:** `MassActionLaw`, stoichiometry, reaction rates and equilibrium in a `Cstr`.

1. Forward reactions in a stirred tank
2. Reversible reactions

```{note}
A closed stirred tank lets us isolate bulk reactions from transport and binding. The mass action law relates reaction rates to reactant concentrations. We use `MassActionLaw` for forward and reversible bulk reactions, then compare them with adsorption: reaction converts species, while binding transfers them between phases.
```

The tank has no flow and no stationary phase. The complete example is shown in code; the next notebook supplies an independent batch-tank checkpoint for the optional exercises.

## Forward reactions
As a simple example, consider the following system:

$$
1\,\mathrm{A} \xrightarrow{k_{AB}} 1\,\mathrm{B}
$$

First, create a `ComponentSystem` with components `A` and `B`.

In [ ]:
from CADETProcess.processModel import ComponentSystem
component_system = ComponentSystem(['A', 'B'])

Next, configure the [MassActionLaw reaction model](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.MassActionLaw.html).
Instantiate it by passing the `ComponentSystem`.
The reaction is added with [`add_reaction()`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.MassActionLaw.add_reaction.html) with these arguments:

- `components`: Names of the participating components, e.g. `["A", "B"]`
- `coefficients`: Stoichiometric coefficients in that same order; negative for reactants, positive for products
- `k_fwd`: Forward reaction rate
- `k_bwd`: Backward reaction rate

In [ ]:
from CADETProcess.processModel import MassActionLaw
reaction_system = MassActionLaw(component_system)
reaction_system.add_reaction(
    components=["A", "B"],
    coefficients=[-1, 1],
    k_fwd=0.1,
    k_bwd=0
)

To demonstrate this reaction, create a `Cstr` and add the reaction to it.
Next come the initial conditions.
The `Cstr` supports reactions in bulk and particle pore phases.
Since the porosity is $1$ by default, only the bulk phase is considered.

In [ ]:
from CADETProcess.processModel import Cstr

reactor = Cstr(component_system, 'reactor')
reactor.bulk_reaction_model = reaction_system
reactor.init_liquid_volume = 1e-6
reactor.const_solid_volume = 0
reactor.c = [1.0, 0.0]

The reactor goes into a `FlowSheet`, and a `Process` is set up from it.
The flow sheet contains only the tank; no timed events are needed.

In [ ]:
from CADETProcess.processModel import FlowSheet
flow_sheet = FlowSheet(component_system)
flow_sheet.add_unit(reactor)

from CADETProcess.processModel import Process
process = Process(flow_sheet, 'reaction_demo')
process.cycle_time = 100

After simulation, plot the results:

In [ ]:
from CADETProcess.simulator import Cadet
simulator = Cadet()
sim_results = simulator.simulate(process)
_ = sim_results.solution.reactor.outlet.plot()

## Reversible reactions
A reversible reaction includes a backward rate. The forward and backward rates determine the equilibrium composition:

$$
2\,\mathrm{A} \xleftrightarrow[k_{BA}]{k_{AB}} \mathrm{B}
$$

Reuse the same units, flow sheet, and process defined above.

In [ ]:
reaction_system = MassActionLaw(component_system)
reaction_system.add_reaction(
    components=["A", "B"],
    coefficients=[-2, 1],
    k_fwd=0.2,
    k_bwd=0.1
)

reactor.bulk_reaction_model = reaction_system

After simulation, plot the results:

In [ ]:
sim_results = simulator.simulate(process)
_ = sim_results.solution.reactor.outlet.plot()

## Summary

- `MassActionLaw.add_reaction()` defines participating components, stoichiometry and rates.
- A closed `Cstr` isolates reaction behavior from flow and binding.
- Reverse rates change the equilibrium composition; binding instead transfers components between phases.

**Now:** [**99.3 Reactions: Exercises**](99.3b_reactions_exercises.ipynb) (optional): add an intermediate and compare reversible reactions.